# Panel de balance — agregados institucionales

Construye `data/interim/paneles_largos/panel_balance_agregados.parquet` con los códigos `AAxxx` (sistema financiero, bancos públicos, bancos privados, top-10, etc.). Listado completo en `dim_grupos`.

**Diferencia importante con el panel principal**: el archivo `bal_hist/balhist.txt` NO incluye los `AAxxx` — solo trae entidades individuales. Los agregados pre-calculados por el BCRA viven en `balres/AAxxx.txt` (balance resumido), con 5 fechas por dump. Se apilan los 133 dumps, deduplicando por dump más reciente.

El panel resultante es de granularidad menor (balance "resumido" en capítulos agregados, no plan completo de cuentas), con frecuencia efectiva trimestral.

Decisión metodológica (§3.5): los agrupamientos viven separados del panel principal para no contaminar promedios y filtros. Uso típico: validación cruzada (la suma de los bancos individuales por grupo debería dar el agregado reportado).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb

IEF_ROOT = RAW / "bcra_ief"
OUT = PANELES / "panel_balance_agregados.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)

BALRES_FECHA_COLS = ["v1", "v2", "v3", "v4", "v5"]
ENTIDAD_FECHA_IDX = list(range(12, 17))  # cols 13-17 (1-based) en entidad/cod.txt

## Helpers

Para conocer las 5 fechas correspondientes a las 5 columnas de balres, el BCRA expone el archivo `entidad/{codigo}.txt`. Para los `AA` codes existe `entidad/AAxxx.txt` análogo a las entidades.

In [2]:
def leer_fechas(entidad_file):
    texto = entidad_file.read_text(encoding="ISO-8859-1")
    campos = [c.strip().strip('"') for c in texto.split("\t")]
    if len(campos) <= ENTIDAD_FECHA_IDX[-1]:
        return []
    return [campos[i] for i in ENTIDAD_FECHA_IDX]

def leer_balres(balres_file, fechas_5):
    cols = ["codigo_entidad", "nombre_entidad", "fecha_corte", "codigo_linea",
            "denominacion_cuenta", "v1", "v2", "v3", "v4", "v5"]
    df = pd.read_csv(balres_file, sep="\t", header=None, names=cols,
                     encoding="ISO-8859-1", dtype=str)
    for c in BALRES_FECHA_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    mapeo = dict(zip(BALRES_FECHA_COLS, fechas_5))
    long = df.melt(
        id_vars=["codigo_entidad", "nombre_entidad", "codigo_linea", "denominacion_cuenta"],
        value_vars=BALRES_FECHA_COLS, var_name="slot_fecha", value_name="saldo_miles_pesos"
    )
    long["yyyymm"] = long["slot_fecha"].map(mapeo)
    long = long.drop(columns=["slot_fecha"]).dropna(subset=["yyyymm"])
    return long

## Apilado de los 133 dumps

In [3]:
dumps = sorted([d for d in IEF_ROOT.iterdir() if d.is_dir() and d.name.isdigit()])
bloques = []
for d in dumps:
    yyyymm_dump = d.name
    balres_dir = d / "Entfin/Tec_Cont/balres"
    entidad_dir = d / "Entfin/Tec_Cont/entidad"
    if not balres_dir.exists() or not entidad_dir.exists():
        continue
    for balres_file in balres_dir.glob("AA*.txt"):
        cod = balres_file.stem
        entidad_file = entidad_dir / f"{cod}.txt"
        if not entidad_file.exists():
            continue
        try:
            fechas = leer_fechas(entidad_file)
            if len(fechas) != 5:
                continue
            long = leer_balres(balres_file, fechas)
            long["dump_yyyymm"] = int(yyyymm_dump)
            bloques.append(long)
        except Exception:
            pass

raw = pd.concat(bloques, ignore_index=True)
print(f"Filas apiladas: {len(raw):,}")

Filas apiladas: 386,635


## Deduplicación y persistencia

In [4]:
raw = raw[raw["yyyymm"].astype(str).str.match(r"^\d{6}$")].copy()
raw["yyyymm"] = raw["yyyymm"].astype(int)
raw["saldo"] = raw["saldo_miles_pesos"] * 1000
raw = raw.sort_values(["codigo_entidad", "yyyymm", "codigo_linea", "dump_yyyymm"])
panel = raw.drop_duplicates(subset=["codigo_entidad", "yyyymm", "codigo_linea"], keep="last")
panel = panel[["codigo_entidad", "nombre_entidad", "yyyymm", "codigo_linea",
               "denominacion_cuenta", "saldo", "dump_yyyymm"]]
panel.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")
print(f"Filas: {len(panel):,}")

Escrito: data/interim/paneles_largos/panel_balance_agregados.parquet
Filas: 86,474


## Validaciones

In [5]:
assert panel["codigo_entidad"].str.startswith("AA").all()
assert panel[["codigo_entidad", "yyyymm", "codigo_linea"]].duplicated().sum() == 0
print("Validaciones OK")
print(f"  agrupamientos: {panel['codigo_entidad'].nunique()}")
print(f"  códigos de línea: {panel['codigo_linea'].nunique()}")
print(f"  rango fechas: {panel['yyyymm'].min()} - {panel['yyyymm'].max()}")

Validaciones OK
  agrupamientos: 12
  códigos de línea: 82
  rango fechas: 201512 - 202601


## Muestra: activo del sistema financiero

In [6]:
duckdb.sql(f"""
    select yyyymm, denominacion_cuenta, saldo / 1e12 as saldo_billones_pesos
    from '{OUT}'
    where codigo_entidad = 'AA000' and denominacion_cuenta like '%C T I V O%'
    order by yyyymm desc
    limit 10
""").df()

,yyyymm,denominacion_cuenta,saldo_billones_pesos
0,202601,A C T I V O,304.462074
1,202512,A C T I V O,303.007518
2,202511,A C T I V O,294.745435
3,202510,A C T I V O,283.984930
4,202509,A C T I V O,277.296210
5,202508,A C T I V O,267.240731
6,202507,A C T I V O,262.162913
7,202506,A C T I V O,252.088008
8,202505,A C T I V O,243.626290
9,202504,A C T I V O,234.815857
